In [9]:
dataset_path = "/kaggle/input/plantvillage/dataset/train"

In [2]:
import os

base_path = "/kaggle/input/plantvillage/dataset"

splits = ["train", "test", "validation"]

all_classes = set()  # use set to avoid duplicates

for split in splits:
    split_path = os.path.join(base_path, split)
    if os.path.exists(split_path):
        classes = os.listdir(split_path)
        all_classes.update(classes)

# Filter only Tomato classes
tomato_classes = sorted([c for c in all_classes if c.startswith("Tomato")])

print("Tomato Classes:")
for c in tomato_classes:
    print(c)

print("Total Tomato Classes:", len(tomato_classes))


Tomato Classes:
Tomato_Bacterial_spot
Tomato_Early_blight
Tomato_Late_blight
Tomato_Leaf_Mold
Tomato_Septoria_leaf_spot
Tomato_Spider_mites_Two_spotted_spider_mite
Tomato__Target_Spot
Tomato__Tomato_YellowLeaf__Curl_Virus
Tomato__Tomato_mosaic_virus
Tomato_healthy
Total Tomato Classes: 10


In [3]:
import os
import shutil

base_dataset_path = "/kaggle/input/plantvillage/dataset"
splits = ["train", "test", "validation"]

tomato_dataset_path = "/kaggle/working/TomatoDataset"
os.makedirs(tomato_dataset_path, exist_ok=True)

for cls in tomato_classes:
    dst_cls_path = os.path.join(tomato_dataset_path, cls)
    os.makedirs(dst_cls_path, exist_ok=True)

    for split in splits:
        src_cls_path = os.path.join(base_dataset_path, split, cls)

        if os.path.exists(src_cls_path):
            for img in os.listdir(src_cls_path):
                src_img = os.path.join(src_cls_path, img)
                dst_img = os.path.join(dst_cls_path, img)

                # avoid overwrite if same filename exists
                if not os.path.exists(dst_img):
                    shutil.copy(src_img, dst_img)

print("✅ Tomato dataset created using train + test + validation")


✅ Tomato dataset created using train + test + validation


In [4]:
total_images = 0

print("\nImage count per Tomato class:\n")

for cls in sorted(os.listdir(tomato_dataset_path)):
    cls_path = os.path.join(tomato_dataset_path, cls)
    count = len([
        f for f in os.listdir(cls_path)
        if os.path.isfile(os.path.join(cls_path, f))
    ])
    print(f"{cls}: {count}")
    total_images += count

print("\nTotal Tomato images:", total_images)



Image count per Tomato class:

Tomato_Bacterial_spot: 2127
Tomato_Early_blight: 1000
Tomato_Late_blight: 1909
Tomato_Leaf_Mold: 952
Tomato_Septoria_leaf_spot: 1771
Tomato_Spider_mites_Two_spotted_spider_mite: 1676
Tomato__Target_Spot: 1404
Tomato__Tomato_YellowLeaf__Curl_Virus: 3208
Tomato__Tomato_mosaic_virus: 373
Tomato_healthy: 1591

Total Tomato images: 16011


In [5]:
import torchvision.transforms as transforms

basic_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


In [6]:
from torchvision.datasets import ImageFolder

full_dataset = ImageFolder(
    root=tomato_dataset_path,
    transform=basic_transform
)

print("Total samples:", len(full_dataset))
print("Classes:", full_dataset.classes)


Total samples: 16011
Classes: ['Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [7]:
from torch.utils.data import random_split

def split_dataset(dataset, percent):
    train_size = int(percent * len(dataset))
    val_size = len(dataset) - train_size
    return random_split(dataset, [train_size, val_size])


In [8]:
train_set, val_set = split_dataset(full_dataset, 0.10)

print("Train:", len(train_set))
print("Val:", len(val_set))


Train: 1601
Val: 14410


In [9]:
label_percents = [0.01,0.05,0.10,0.20,0.30,0.40,0.50]


In [10]:
results = {
    "supervised": [],
    "simclr": [],
    "byol": [],
    "fixmatch": [],
    "mixmatch": []
}


In [11]:
import torch
import torch.nn as nn
import torchvision.models as models

device = "cuda" if torch.cuda.is_available() else "cpu"

def get_supervised_model(num_classes):
    model = models.resnet18(pretrained=False)
    model.fc = nn.Linear(512, num_classes)
    return model.to(device)


In [12]:
def train_supervised(train_loader, val_loader, num_classes, epochs=5):
    model = get_supervised_model(num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total



In [13]:
from torch.utils.data import DataLoader

train_set, val_set = split_dataset(full_dataset, 0.10)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)

acc = train_supervised(train_loader, val_loader, len(full_dataset.classes))
print("Accuracy:", acc)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Accuracy: 22.15822345593338


In [14]:
# ===============================
# FIXMATCH ON TOMATO DATASET
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset, random_split
from itertools import cycle

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------
# 1. TRANSFORMS (FixMatch style)
# ----------------------------------
weak_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

strong_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor()
])

# ----------------------------------
# 2. LOAD TOMATO DATASET (NO TRANSFORM)
# ----------------------------------
tomato_dataset_path = "/kaggle/working/TomatoDataset"

full_dataset = ImageFolder(
    root=tomato_dataset_path,
    transform=None
)

num_classes = len(full_dataset.classes)
print("Number of Tomato classes:", num_classes)

# ----------------------------------
# 3. SPLIT: LABELED / UNLABELED / VAL
# ----------------------------------
label_percent = 0.10   # 10% labeled

num_total = len(full_dataset)
num_labeled = int(label_percent * num_total)
num_unlabeled = int(0.80 * num_total)
num_val = num_total - num_labeled - num_unlabeled

labeled_set, unlabeled_set, val_set = random_split(
    full_dataset,
    [num_labeled, num_unlabeled, num_val]
)

print("Labeled:", len(labeled_set))
print("Unlabeled:", len(unlabeled_set))
print("Validation:", len(val_set))

# ----------------------------------
# 4. DATASET WRAPPERS
# ----------------------------------
class LabeledWrapper(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label


class UnlabeledWrapper(Dataset):
    def __init__(self, subset, weak_t, strong_t):
        self.subset = subset
        self.weak_t = weak_t
        self.strong_t = strong_t

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, _ = self.subset[idx]
        return self.weak_t(img), self.strong_t(img)

# ----------------------------------
# 5. DATALOADERS
# ----------------------------------
train_labeled_loader = DataLoader(
    LabeledWrapper(labeled_set, weak_transform),
    batch_size=32,
    shuffle=True
)

train_unlabeled_loader = DataLoader(
    UnlabeledWrapper(unlabeled_set, weak_transform, strong_transform),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    LabeledWrapper(val_set, weak_transform),
    batch_size=32
)

# ----------------------------------
# 6. MODEL (FROM SCRATCH)
# ----------------------------------
model = models.resnet18(weights=None)
model.fc = nn.Linear(512, num_classes)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# FixMatch hyperparameters
confidence_threshold = 0.80
lambda_u = 1.0
epochs = 15
supervised_warmup_epochs = 5

# ----------------------------------
# 7. FIXMATCH TRAINING LOOP
# ----------------------------------
unlabeled_iter = cycle(train_unlabeled_loader)

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images_l, labels_l in train_labeled_loader:
        images_l, labels_l = images_l.to(device), labels_l.to(device)

        # ----- Supervised loss -----
        logits_l = model(images_l)
        loss_l = criterion(logits_l, labels_l)

        # ----- Unlabeled FixMatch loss -----
        images_w, images_s = next(unlabeled_iter)
        images_w, images_s = images_w.to(device), images_s.to(device)

        with torch.no_grad():
            logits_w = model(images_w)
            probs = F.softmax(logits_w, dim=1)
            max_probs, pseudo_labels = torch.max(probs, dim=1)
            mask = max_probs.ge(confidence_threshold).float()

        logits_s = model(images_s)
        loss_u = (criterion(logits_s, pseudo_labels) * mask).mean()

        # ----- Warm-up logic -----
        if epoch < supervised_warmup_epochs:
            loss = loss_l
        else:
            loss = loss_l + lambda_u * loss_u

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] FixMatch Loss: {total_loss/len(train_labeled_loader):.4f}")

# ----------------------------------
# 8. EVALUATION
# ----------------------------------
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

fixmatch_acc = 100 * correct / total
print("\n✅ FixMatch Accuracy (Tomato-only, no SSL):", fixmatch_acc)


Number of Tomato classes: 10
Labeled: 1601
Unlabeled: 12808
Validation: 1602
Epoch [1/15] FixMatch Loss: 1.5938
Epoch [2/15] FixMatch Loss: 1.1259
Epoch [3/15] FixMatch Loss: 1.0372
Epoch [4/15] FixMatch Loss: 0.8678
Epoch [5/15] FixMatch Loss: 0.7723
Epoch [6/15] FixMatch Loss: 1.5181
Epoch [7/15] FixMatch Loss: 1.3257
Epoch [8/15] FixMatch Loss: 1.3070
Epoch [9/15] FixMatch Loss: 1.2630
Epoch [10/15] FixMatch Loss: 1.2689
Epoch [11/15] FixMatch Loss: 1.1730
Epoch [12/15] FixMatch Loss: 1.1940
Epoch [13/15] FixMatch Loss: 1.1062
Epoch [14/15] FixMatch Loss: 1.0210
Epoch [15/15] FixMatch Loss: 1.0365

✅ FixMatch Accuracy (Tomato-only, no SSL): 73.40823970037454


In [15]:
# ===============================
# MIXMATCH ON TOMATO DATASET
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset, random_split
from itertools import cycle

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------
# 1. TRANSFORMS
# ----------------------------------
weak_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

strong_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor()
])

# ----------------------------------
# 2. LOAD TOMATO DATASET
# ----------------------------------
tomato_dataset_path = "/kaggle/working/TomatoDataset"

full_dataset = ImageFolder(
    root=tomato_dataset_path,
    transform=None
)

num_classes = len(full_dataset.classes)
print("Number of Tomato classes:", num_classes)

# ----------------------------------
# 3. SPLIT DATASET
# ----------------------------------
label_percent = 0.10

num_total = len(full_dataset)
num_labeled = int(label_percent * num_total)
num_unlabeled = int(0.80 * num_total)
num_val = num_total - num_labeled - num_unlabeled

labeled_set, unlabeled_set, val_set = random_split(
    full_dataset,
    [num_labeled, num_unlabeled, num_val]
)

print("Labeled:", len(labeled_set))
print("Unlabeled:", len(unlabeled_set))
print("Validation:", len(val_set))

# ----------------------------------
# 4. DATASET WRAPPERS
# ----------------------------------
class LabeledWrapper(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label


class UnlabeledWrapper(Dataset):
    def __init__(self, subset, weak_t, strong_t):
        self.subset = subset
        self.weak_t = weak_t
        self.strong_t = strong_t

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, _ = self.subset[idx]
        u1 = self.weak_t(img)
        u2 = self.strong_t(img)
        return u1, u2

# ----------------------------------
# 5. DATALOADERS
# ----------------------------------
train_labeled_loader = DataLoader(
    LabeledWrapper(labeled_set, weak_transform),
    batch_size=32,
    shuffle=True
)

train_unlabeled_loader = DataLoader(
    UnlabeledWrapper(unlabeled_set, weak_transform, strong_transform),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    LabeledWrapper(val_set, weak_transform),
    batch_size=32
)

# ----------------------------------
# 6. MODEL
# ----------------------------------
model = models.resnet18(weights=None)
model.fc = nn.Linear(512, num_classes)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# MixMatch hyperparameters
lambda_u = 1.0
epochs = 15
warmup_epochs = 5
temperature = 0.5

# ----------------------------------
# 7. SHARPEN FUNCTION
# ----------------------------------
def sharpen(p, T=0.5):
    p = p ** (1 / T)
    return p / p.sum(dim=1, keepdim=True)

# ----------------------------------
# 8. MIXMATCH TRAINING LOOP
# ----------------------------------
unlabeled_iter = cycle(train_unlabeled_loader)

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images_l, labels_l in train_labeled_loader:
        images_l, labels_l = images_l.to(device), labels_l.to(device)

        # ----- supervised loss -----
        logits_l = model(images_l)
        loss_l = criterion(logits_l, labels_l)

        # ----- MixMatch unlabeled loss -----
        u1, u2 = next(unlabeled_iter)
        u1, u2 = u1.to(device), u2.to(device)

        with torch.no_grad():
            p1 = F.softmax(model(u1), dim=1)
            p2 = F.softmax(model(u2), dim=1)
            p_avg = (p1 + p2) / 2
            p_sharp = sharpen(p_avg, temperature)

        logits_u = model(u1)
        loss_u = F.mse_loss(F.softmax(logits_u, dim=1), p_sharp)

        # ----- warm-up -----
        if epoch < warmup_epochs:
            loss = loss_l
        else:
            loss = loss_l + lambda_u * loss_u

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] MixMatch Loss: {total_loss/len(train_labeled_loader):.4f}")

# ----------------------------------
# 9. EVALUATION
# ----------------------------------
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

mixmatch_acc = 100 * correct / total
print("\n✅ MixMatch Accuracy (Tomato-only, no SSL):", mixmatch_acc)


Number of Tomato classes: 10
Labeled: 1601
Unlabeled: 12808
Validation: 1602
Epoch [1/15] MixMatch Loss: 1.4902
Epoch [2/15] MixMatch Loss: 1.1378
Epoch [3/15] MixMatch Loss: 0.9950
Epoch [4/15] MixMatch Loss: 0.9426
Epoch [5/15] MixMatch Loss: 0.8070
Epoch [6/15] MixMatch Loss: 0.7149
Epoch [7/15] MixMatch Loss: 0.7112
Epoch [8/15] MixMatch Loss: 0.6914
Epoch [9/15] MixMatch Loss: 0.6440
Epoch [10/15] MixMatch Loss: 0.5238
Epoch [11/15] MixMatch Loss: 0.5110
Epoch [12/15] MixMatch Loss: 0.5472
Epoch [13/15] MixMatch Loss: 0.4955
Epoch [14/15] MixMatch Loss: 0.4995
Epoch [15/15] MixMatch Loss: 0.4878

✅ MixMatch Accuracy (Tomato-only, no SSL): 79.15106117353308
